# CESA3 — Why do T1–T4 look structurally different in VMD?

Whole-protein RMSD (see `rmsd_merged_complex.ipynb`) tells you *how much*
the trajectories differ overall, but not *why*. This notebook adds two
complementary, per-residue diagnostics to localize the source of the
apparent incompatibility you're seeing visually in VMD:

1. **Per-residue RMSF within each trajectory** — each T (T1–T4) compared
   only to its own frame 0. Shows which residues are simply flexible
   *within* a given run (loops, termini, etc.) — not yet a comparison
   *between* runs.

2. **Inter-trajectory structural divergence per residue** — compares the
   time-averaged structure of each trajectory against the others. This is
   the piece that directly explains *why* T1–T4 look different from each
   other: if divergence is concentrated in a few residues, it's a local/
   flexible-region story; if it's spread across the whole backbone, it's
   a genuine global conformational difference between runs.

**Inputs (already produced by `process_trajectories_CESA3.sh`):**
- `R1/T1/backbone.xtc`, `R1/T2/backbone.xtc`, `R1/T3/backbone.xtc`, `R1/T4/backbone.xtc`
- `R1/backbone.pdb` (topology — same atom order/count as all backbone.xtc files)

**Outputs (written next to this notebook):**
- `rmsf_per_trajectory.csv` — per-residue RMSF, one column per T
- `rmsf_per_trajectory.png` — overlaid RMSF curves, T1–T4
- `interT_divergence_per_residue.csv` — per-residue spread across T1–T4 average structures
- `interT_divergence_per_residue.png` — bar/line plot highlighting divergent residues
- `interT_pairwise_rmsd.csv` — pairwise RMSD between each T's averaged structure

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

print('MDAnalysis', mda.__version__)

## 1. Configuration

In [ ]:
# BASE_DIR = the process_CESA3 folder containing R1/
BASE_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3')

TOP_PDB     = BASE_DIR / 'R1' / 'backbone.pdb'
TRAJECTORIES = ['T1', 'T2', 'T3', 'T4']
TRAJ_FMT    = 'R1/{T}/backbone.xtc'
ATOM_SEL    = 'backbone'

OUT_DIR = BASE_DIR
OUT_RMSF_CSV   = OUT_DIR / 'rmsf_per_trajectory.csv'
OUT_RMSF_PNG   = OUT_DIR / 'rmsf_per_trajectory.png'
OUT_DIV_CSV    = OUT_DIR / 'interT_divergence_per_residue.csv'
OUT_DIV_PNG    = OUT_DIR / 'interT_divergence_per_residue.png'
OUT_PAIRWISE_CSV = OUT_DIR / 'interT_pairwise_rmsd.csv'

assert TOP_PDB.exists(), f'Topology not found: {TOP_PDB}'
traj_paths = {}
for T in TRAJECTORIES:
    p = BASE_DIR / TRAJ_FMT.format(T=T)
    assert p.exists(), f'Missing trajectory: {p}'
    traj_paths[T] = p
    print(f'  {T}: {p.relative_to(BASE_DIR)}  ({p.stat().st_size / 1e6:.1f} MB)')
print(f'Topology: {TOP_PDB.relative_to(BASE_DIR)}')

## 2. Per-residue RMSF within each trajectory

For each T, frames are first aligned (rot+trans) to that trajectory's own
frame 0 on the backbone selection, then per-residue RMSF is computed.
This is purely a within-run flexibility measure — T1's RMSF curve does
NOT yet tell you anything about how T1 compares to T2.

In [ ]:
import tempfile

def compute_rmsf(top_pdb, traj_path, sel='backbone'):
    """Memory-safe version: aligns to a temporary file on disk instead of
    loading the whole trajectory into RAM (avoids OOM kernel crashes on
    long trajectories).

    IMPORTANT: groups by (segid/chain, resid) rather than resid alone.
    This PDB's residue numbering is NOT unique across chains (e.g. chain A
    residues 254-619 overlap numerically with chain B residues 254-619),
    so grouping by resid alone would silently average together residues
    from different chains/proteins."""
    u = mda.Universe(str(top_pdb), str(traj_path))
    atoms = u.select_atoms(sel)

    with tempfile.NamedTemporaryFile(suffix='.xtc', delete=False) as tmp:
        tmp_path = tmp.name

    try:
        # in_memory=False (default) streams frame-by-frame to disk instead
        # of holding the full trajectory as a NumPy array in RAM.
        align.AlignTraj(u, u, select=sel, ref_frame=0,
                         filename=tmp_path, in_memory=False).run()

        u_aligned = mda.Universe(str(top_pdb), tmp_path)
        atoms_aligned = u_aligned.select_atoms(sel)
        rmsf_calc = rms.RMSF(atoms_aligned).run()
        df = pd.DataFrame({
            'chain'  : atoms_aligned.chainIDs,
            'resid'  : atoms_aligned.resids,
            'rmsf_A' : rmsf_calc.results.rmsf,
        })
        return df.groupby(['chain', 'resid'], as_index=False)['rmsf_A'].mean()
    finally:
        Path(tmp_path).unlink(missing_ok=True)

rmsf_per_T = {}
for T in TRAJECTORIES:
    print(f'-- {T}: computing RMSF (aligned to its own frame 0)')
    rmsf_per_T[T] = compute_rmsf(TOP_PDB, traj_paths[T], sel=ATOM_SEL)

# Merge into one wide table on (chain, resid), since resid alone is not
# unique across chains in this structure (A/B/C/D overlap numerically).
df_rmsf = rmsf_per_T['T1'][['chain', 'resid']].copy()
for T in TRAJECTORIES:
    df_rmsf[f'rmsf_{T}_A'] = rmsf_per_T[T]['rmsf_A'].values

# Build a chain-aware label for plotting (e.g. "A:254", "B:254") so
# overlapping resid numbers from different chains never collide on the
# x-axis, and a continuous global index for a clean, gap-free x-axis.
df_rmsf['label'] = df_rmsf['chain'] + ':' + df_rmsf['resid'].astype(str)
df_rmsf = df_rmsf.sort_values(['chain', 'resid']).reset_index(drop=True)
df_rmsf['global_index'] = np.arange(len(df_rmsf))

# Chain boundaries (for plotting vertical separators / labels)
chain_bounds = df_rmsf.groupby('chain')['global_index'].agg(['min', 'max'])
print('Chain ranges in this structure:')
print(chain_bounds)

df_rmsf.to_csv(OUT_RMSF_CSV, index=False)
print(f'Wrote {OUT_RMSF_CSV.name}')
df_rmsf.head()

### Plot — RMSF per residue, overlaid across T1–T4

Residues where one trajectory's curve spikes much higher than the others
are flexible specifically in that run — worth a closer look in VMD at
that residue range.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
colors = {'T1': '#1f77b4', 'T2': '#2ca02c', 'T3': '#d62728', 'T4': '#9467bd'}

for T in TRAJECTORIES:
    ax.plot(df_rmsf['global_index'], df_rmsf[f'rmsf_{T}_A'],
            label=T, color=colors[T], lw=1.1, alpha=0.85)

# Mark chain boundaries and label each chain
y_top = df_rmsf[[f'rmsf_{T}_A' for T in TRAJECTORIES]].values.max() * 1.05
for chain, row in chain_bounds.iterrows():
    if chain != chain_bounds.index[0]:
        ax.axvline(row['min'] - 0.5, color='grey', ls='--', lw=0.8, alpha=0.6)
    x_center = (row['min'] + row['max']) / 2
    ax.text(x_center, y_top, f'Chain {chain}', ha='center', va='bottom',
            fontsize=10, color='grey')

ax.set_xlabel('Residue (chain-aware index; numbering resets per chain in this PDB)')
ax.set_ylabel('RMSF (A)')
ax.set_title('CESA3 backbone RMSF per residue — within-trajectory flexibility (T1\u2013T4)')
ax.legend(frameon=False, ncol=4, fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_RMSF_PNG, dpi=200)
print(f'Wrote {OUT_RMSF_PNG.name}')
plt.show()

## 3. Inter-trajectory structural divergence per residue

This is the analysis that directly addresses *why T1–T4 look different
from each other in VMD*. For each trajectory, we compute its
**time-averaged structure** (mean position of each atom across all frames,
after aligning to a common reference). Then, per residue, we measure how
much those 4 average structures disagree with each other.

A residue with high inter-T spread means: across your 4 independent runs,
that part of the protein settled into noticeably different average
positions — a real, localized structural divergence between runs, as
opposed to within-run wobble (already captured by RMSF above).

In [ ]:
def average_structure(top_pdb, traj_path, ref_universe, sel='backbone'):
    """Align trajectory to a common reference, then return the mean
    position of each selected atom across all frames. Streams to a
    temp file on disk rather than loading the full trajectory into RAM.
    Returns chain IDs alongside resids, since resid alone is not unique
    across chains in this structure."""
    u = mda.Universe(str(top_pdb), str(traj_path))

    with tempfile.NamedTemporaryFile(suffix='.xtc', delete=False) as tmp:
        tmp_path = tmp.name

    try:
        align.AlignTraj(u, ref_universe, select=sel,
                         filename=tmp_path, in_memory=False).run()

        u_aligned = mda.Universe(str(top_pdb), tmp_path)
        atoms = u_aligned.select_atoms(sel)

        coords_sum = np.zeros((len(atoms), 3))
        n = 0
        for ts in u_aligned.trajectory:
            coords_sum += atoms.positions
            n += 1
        return atoms.chainIDs, atoms.resids, coords_sum / n
    finally:
        Path(tmp_path).unlink(missing_ok=True)

# Common reference = R1/backbone.pdb (T1, frame 0) — same reference used
# throughout the rest of the pipeline (steps 9-10, rmsd_merged_complex.ipynb)
ref_u = mda.Universe(str(TOP_PDB))

avg_coords = {}
chains_ref = None
resids_ref = None
for T in TRAJECTORIES:
    print(f'-- {T}: computing time-averaged structure (aligned to R1/backbone.pdb)')
    chains, resids, coords = average_structure(TOP_PDB, traj_paths[T], ref_u, sel=ATOM_SEL)
    avg_coords[T] = coords
    if chains_ref is None:
        chains_ref = chains
        resids_ref = resids

### Per-residue divergence = spread of the 4 average positions

For each residue, take its average backbone position in T1, T2, T3, T4
(3 coordinates each), then compute the standard deviation across the 4
trajectories as a single scalar (using the RMS of per-atom coordinate
std as a simple, interpretable per-residue divergence score in Angstrom).

In [ ]:
# Stack into shape (n_traj, n_atoms, 3)
stack = np.stack([avg_coords[T] for T in TRAJECTORIES])   # (4, n_atoms, 3)

per_atom_std = stack.std(axis=0)                # (n_atoms, 3)
per_atom_divergence = np.linalg.norm(per_atom_std, axis=1)   # (n_atoms,)

df_atoms = pd.DataFrame({
    'chain': chains_ref,
    'resid': resids_ref,
    'divergence_A': per_atom_divergence,
})
df_divergence = df_atoms.groupby(['chain', 'resid'], as_index=False)['divergence_A'].mean()
df_divergence['label'] = df_divergence['chain'] + ':' + df_divergence['resid'].astype(str)
df_divergence = df_divergence.sort_values(['chain', 'resid']).reset_index(drop=True)
df_divergence['global_index'] = np.arange(len(df_divergence))

div_chain_bounds = df_divergence.groupby('chain')['global_index'].agg(['min', 'max'])

df_divergence.to_csv(OUT_DIV_CSV, index=False)
print(f'Wrote {OUT_DIV_CSV.name}')

# Flag the most divergent residues (now unambiguous: chain + resid)
top_n = 15
top_divergent = df_divergence.sort_values('divergence_A', ascending=False).head(top_n)
print(f'\nTop {top_n} most divergent residues (largest spread of average position across T1-T4):')
print(top_divergent[['label', 'divergence_A']].to_string(index=False))

### Plot — per-residue divergence across T1–T4

Sharp, isolated peaks = a small number of specific residues (likely a
loop, terminus, or flexible domain) are driving the visual difference
you see in VMD. A broad, uniformly elevated profile across most of the
backbone instead suggests a genuine global conformational shift between
runs, not a localized one.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df_divergence['global_index'], df_divergence['divergence_A'],
        color='#d62728', lw=1.2)
ax.fill_between(df_divergence['global_index'], 0, df_divergence['divergence_A'],
                color='#d62728', alpha=0.15)

mean_div = df_divergence['divergence_A'].mean()
ax.axhline(mean_div, color='black', ls='--', lw=0.9, alpha=0.6,
           label=f'mean divergence = {mean_div:.2f} A')

# Mark chain boundaries
y_top = df_divergence['divergence_A'].max() * 1.05
for chain, row in div_chain_bounds.iterrows():
    if chain != div_chain_bounds.index[0]:
        ax.axvline(row['min'] - 0.5, color='grey', ls='--', lw=0.8, alpha=0.6)
    x_center = (row['min'] + row['max']) / 2
    ax.text(x_center, y_top, f'Chain {chain}', ha='center', va='bottom',
            fontsize=10, color='grey')

ax.set_xlabel('Residue (chain-aware index; numbering resets per chain in this PDB)')
ax.set_ylabel('Inter-trajectory divergence (A)')
ax.set_title('CESA3 — per-residue divergence between T1\u2013T4 average structures\n'
             '(high = this residue settled at different positions across runs)')
ax.legend(frameon=False, fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIV_PNG, dpi=200)
print(f'Wrote {OUT_DIV_PNG.name}')
plt.show()

## 4. Pairwise RMSD between trajectories' average structures

A single number per (Ti, Tj) pair: RMSD between their time-averaged
structures (already aligned to the same reference above). This tells
you which specific pair(s) of trajectories differ most from each other
overall — e.g. if T3 is the outlier, T1-T3, T2-T3, T3-T4 will all be
high while T1-T2, T1-T4, T2-T4 stay low.

In [ ]:
pairwise_rows = []
for i, T1 in enumerate(TRAJECTORIES):
    for T2 in TRAJECTORIES[i+1:]:
        diff = avg_coords[T1] - avg_coords[T2]
        rmsd_val = np.sqrt((diff ** 2).sum(axis=1).mean())
        pairwise_rows.append({'pair': f'{T1}-{T2}', 'rmsd_A': rmsd_val})

df_pairwise = pd.DataFrame(pairwise_rows).sort_values('rmsd_A', ascending=False)
df_pairwise.to_csv(OUT_PAIRWISE_CSV, index=False)
print(f'Wrote {OUT_PAIRWISE_CSV.name}')
print('\nPairwise RMSD between average structures (Angstrom), highest first:')
print(df_pairwise.to_string(index=False))

## 5. Summary

Read the results in this order:

1. **Pairwise RMSD (section 4)** — tells you if one trajectory is the
   clear outlier (high RMSD against all others) or if all 4 are mutually
   quite different (no single outlier).
2. **Per-residue divergence plot (section 3)** — tells you *where* in the
   sequence that difference is concentrated. Cross-reference the
   `top_divergent` residue list against the actual structure in VMD/
   PyMOL (e.g. `resid 145-160` if that's what shows up) to see if it
   corresponds to a loop, terminus, or known flexible domain.
3. **RMSF plot (section 2)** — tells you if those same divergent residues
   are *also* highly flexible within a single run (consistent with "it's
   just a floppy loop that settles differently each time") or if they're
   rigid within each run but different *between* runs (more suggestive
   of a real alternate conformation/state being sampled by that run).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis import rms

BASE_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA3/process_CESA3/R1')
REF_PDB = BASE_DIR / 'backbone.pdb'   # same shared reference used throughout

trajectories = {
    'T1': BASE_DIR / 'CESA3_T1_bobina' / 'backbone.xtc',
    'T2': BASE_DIR / 'CESA3_T2_bobina' / 'backbone.xtc',
    'T3': BASE_DIR / 'CESA3_T3_bobina' / 'backbone.xtc',
    'T4': BASE_DIR / 'CESA3_T4_bobina' / 'backbone.xtc',
}

ref = mda.Universe(str(REF_PDB))
results = {}

for name, path in trajectories.items():
    u = mda.Universe(str(REF_PDB), str(path))
    R = rms.RMSD(u, ref, select='backbone').run()
    rmsd_vals = R.results.rmsd[:, 2]
    results[name] = rmsd_vals
    print(f'{name}: n={len(rmsd_vals)}  mean={rmsd_vals.mean():.3f} ± {rmsd_vals.std():.3f} Å  '
          f'range=[{rmsd_vals.min():.3f}, {rmsd_vals.max():.3f}]')

all_vals = np.concatenate(list(results.values()))
print(f'\nOVERALL: n={len(all_vals)}  mean={all_vals.mean():.3f} ± {all_vals.std():.3f} Å  '
      f'range=[{all_vals.min():.3f}, {all_vals.max():.3f}]')